# convtranspose-bn-activation-block — worked example 1: Build the final generator block with Tanh (no BatchNorm)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `convtranspose-bn-activation-block`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The **last** block of a DCGAN generator differs from the inner blocks: it uses `Tanh` instead of `ReLU` and drops `BatchNorm`. Tanh squashes outputs to `[-1, 1]`, matching the `Normalize((0.5,), (0.5,))` preprocessing applied to real images. Because no BN follows the ConvTranspose here, the `bias` term is kept (`bias=True`).

## Worked solution

**Step 1 — Recognize this is the output layer.** Inner generator blocks are `ConvT -> BN -> ReLU`. The final block instead is `ConvT -> Tanh`. There is no BatchNorm because BN would re-center/re-scale the values away from the clean `[-1,1]` range that Tanh produces.

**Step 2 — Keep the bias.** In the inner blocks we set `bias=False` because BN's affine `beta` parameter would make the conv bias redundant. Here there is no BN, so the ConvTranspose keeps its bias (`bias=True`, the default) — leave it on.

**Step 3 — Spatial doubling.** Use the standard DCGAN upsampling settings `kernel_size=4, stride=2, padding=1`. The output-size formula `H_out = (H_in-1)*stride - 2*padding + kernel = (H_in-1)*2 - 2 + 4 = 2*H_in` doubles the spatial dimension. A 16x16 input becomes 32x32.

**Step 4 — Wire it up and check the range.** Compose `nn.Sequential(ConvTranspose2d(...), nn.Tanh())`. After running a random input through it, every output value must lie in `[-1, 1]` thanks to Tanh.

In [ ]:
import torch.nn as nn

def build_final_block(in_channels, out_channels):
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=True),
        nn.Tanh(),
    )

t.manual_seed(0)
block = build_final_block(64, 3)
x = t.randn(2, 64, 16, 16)
out = block(x)
print('output shape:', tuple(out.shape))
print('min, max:', round(out.min().item(), 4), round(out.max().item(), 4))
print('has batchnorm:', any(isinstance(m, nn.BatchNorm2d) for m in block))
print('has tanh:', any(isinstance(m, nn.Tanh) for m in block))